# 08. Grouping and Categorization (Hashing)

Grouping means placing items into collections based on a shared property. A dictionary is a natural fit:

`group_key -> list of items`

Examples: group words by anagram signature, group numbers by remainder, or group records by category.

**Core question:** What property makes two items belong to the same group?

## 1. Understand the Grouping Pattern

Suppose we have words: `['eat', 'tea', 'tan', 'ate', 'nat', 'bat']`.

We want words with the same letters to be in the same group.

A dictionary can store:

- Key: a signature that represents the group.
- Value: a list containing all items with that signature.

The hardest part is usually designing the correct group key.

## 2. Basic Grouping with a Dictionary

Use a dictionary where each key maps to a list. `setdefault(key, [])` creates an empty list when the key is missing, then returns the list so we can append.

Alternative: use `defaultdict(list)` from `collections`. Understand `setdefault` first because it makes the dictionary behavior visible.

In [1]:
items = ['apple', 'banana', 'apricot', 'blueberry', 'avocado']
groups = {}

for word in items:
    key = word[0]  # group by first character
    groups.setdefault(key, []).append(word)

print(groups)

{'a': ['apple', 'apricot', 'avocado'], 'b': ['banana', 'blueberry']}


## 3. Pattern 1 — Group Anagrams

Two words are anagrams if they contain the same characters with the same frequencies.

A simple group key is the sorted version of the word:

`key = ''.join(sorted(word))`

For example, `eat`, `tea`, and `ate` all produce `aet`.

Algorithm:
1. Create an empty dictionary.
2. For each word, calculate its signature.
3. Append the word to the list for that signature.
4. Return the dictionary's values.

If there are n words of maximum length L, sorting each word costs O(L log L), so total time is approximately O(n * L log L).

In [2]:
def group_anagrams(words):
    groups = {}

    for word in words:
        key = ''.join(sorted(word))
        groups.setdefault(key, []).append(word)

    return list(groups.values())

words = ['eat', 'tea', 'tan', 'ate', 'nat', 'bat']
print(group_anagrams(words))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


## 4. Group Anagrams — Frequency Signature

If the input contains only lowercase English letters, we can use a 26-element frequency tuple as the key instead of sorting each word.

For each character, increment its alphabet position. Convert the list to a tuple because dictionary keys must be hashable.

For n words of maximum length L, building signatures takes O(n * L + 26n), which is O(n * (L + 26)). With a fixed alphabet, this is often described as O(n * L).

In [3]:
def group_anagrams_counting(words):
    groups = {}

    for word in words:
        count = [0] * 26
        for ch in word:
            count[ord(ch) - ord('a')] += 1

        key = tuple(count)
        groups.setdefault(key, []).append(word)

    return list(groups.values())

print(group_anagrams_counting(['eat', 'tea', 'tan', 'ate', 'nat', 'bat']))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


## 5. Choosing a Group Key

A good group key must satisfy this rule:

**Items that belong in the same group must produce equal keys; items from different groups must not accidentally produce the same key.**

Examples:

- Anagrams: sorted characters or character-frequency tuple.
- Same remainder: `number % k`.
- Same first letter: `word[0]` (handle empty strings if allowed).
- Same normalized form: a carefully defined lowercase/trimmed representation.

Do not use a key that loses information needed to distinguish groups.

In [4]:
nums = [10, 11, 12, 13, 14, 15, 16]
k = 3
groups = {}

for num in nums:
    key = num % k
    groups.setdefault(key, []).append(num)

print(groups)  # grouped by remainder

{1: [10, 13, 16], 2: [11, 14], 0: [12, 15]}


## 6. Pattern 2 — Group by a Computed Property

The key does not have to be an element from the input. It can be calculated from each item.

Examples:

- Group numbers by parity: `num % 2`.
- Group numbers by remainder: `num % k`.
- Group words by length: `len(word)`.
- Group records by a field: `record['department']`.

The dictionary stores `computed_property -> items_with_that_property`.

In [5]:
words = ['hi', 'cat', 'to', 'apple', 'dog', 'sun']
groups = {}

for word in words:
    key = len(word)
    groups.setdefault(key, []).append(word)

print(groups)

{2: ['hi', 'to'], 3: ['cat', 'dog', 'sun'], 5: ['apple']}


## 7. Grouping with defaultdict(list)

`defaultdict(list)` automatically creates an empty list for a missing key.

This avoids writing `setdefault` on every insertion. Both approaches express the same grouping pattern.

In [6]:
from collections import defaultdict

words = ['eat', 'tea', 'tan', 'ate', 'nat', 'bat']
groups = defaultdict(list)

for word in words:
    key = ''.join(sorted(word))
    groups[key].append(word)

print(list(groups.values()))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


## 8. Grouping vs Counting

These are related but different:

- **Counting:** `key -> number of occurrences`.
- **Grouping:** `key -> list of original items`.

If a problem asks only how many items belong to each category, store an integer count. If it asks to return the actual members, store a list.

Do not store full lists when counts alone are enough; that wastes memory.

In [7]:
items = ['red', 'blue', 'red', 'green', 'blue', 'red']

# Count only
counts = {}
for item in items:
    counts[item] = counts.get(item, 0) + 1

# Store all members (same category example)
groups = {}
for item in items:
    groups.setdefault(item, []).append(item)

print(counts)
print(groups)

{'red': 3, 'blue': 2, 'green': 1}
{'red': ['red', 'red', 'red'], 'blue': ['blue', 'blue'], 'green': ['green']}


## 9. Common Mistakes

1. Choosing a group key that does not uniquely represent the required property.
2. Using a mutable list as a dictionary key; use a tuple for frequency signatures.
3. Sorting each word when a fixed-alphabet frequency signature is more suitable.
4. Using a frequency count when the problem asks for the actual grouped items.
5. Forgetting case sensitivity: `Eat` and `tea` are not anagrams under exact character comparison unless normalized.
6. Forgetting spaces or punctuation if the problem's definition says to ignore them.
7. Assuming dictionary group order is the same as sorted order. Python preserves insertion order, but that does not mean groups are sorted by key.

## 10. Complexity Analysis

Let n be the number of items and L the maximum item length.

| Strategy | Approximate time | Extra space |
|---|---:|---:|
| Group by simple property | O(n) average, excluding key computation | O(n) for stored groups |
| Group anagrams by sorting | O(n * L log L) | O(n * L) for groups and keys |
| Group lowercase anagrams by counts | O(n * (L + 26)) | O(n * 26 + n * L) including groups |

Hash map operations are O(1) average. Key construction cost must also be included.

## 11. Interview Thinking Checklist

1. What defines membership in the same group?
2. Can I compute a canonical key for each item?
3. Is the key hashable? If not, can I convert it to a tuple or string?
4. Do I need counts or the actual items?
5. Can I build each key in linear time, or am I sorting?
6. Are case, spaces, punctuation, or input constraints important?
7. What is the total cost of key construction plus dictionary operations?

A strong explanation: "I map each item to a canonical group key. Items with equal keys are appended to the same list. This processes every item once, plus the cost of constructing its key."

## 12. Mini Exercises

Try these without looking up solutions:

1. Group words by their first character.
2. Group integers by even/odd.
3. Group words by length.
4. Group anagrams using sorted-string keys.
5. Group lowercase anagrams using a 26-count tuple.
6. Explain why a tuple can be a dictionary key but a list cannot.
7. Change a grouping solution to return counts instead of lists.

**Challenge:** Given a list of strings, group together strings that are anagrams. Preserve the original order of words inside each group.

In [8]:
def group_anagrams_challenge(words):
    groups = {}

    for word in words:
        # TODO: create a canonical key
        # TODO: append the word to the correct group
        pass

    return list(groups.values())


words = ['abc', 'bca', 'dog', 'cab', 'god']
print(group_anagrams_challenge(words))

[]


## 13. Reusable Templates and Revision

```python
# General grouping
groups = {}
for item in items:
    key = make_key(item)
    groups.setdefault(key, []).append(item)

# Grouping with defaultdict
from collections import defaultdict
groups = defaultdict(list)
for item in items:
    key = make_key(item)
    groups[key].append(item)

# Group anagrams by sorted signature
key = ''.join(sorted(word))

# Group lowercase anagrams by frequency signature
count = [0] * 26
for ch in word:
    count[ord(ch) - ord('a')] += 1
key = tuple(count)
```

**Key idea:** design a canonical, hashable key that represents the property shared by all items in a group.

**Next concept:** Anagram Patterns — deeper problem-solving with frequency maps and grouping.